# 第42章 子图与组合图（subplots）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 11 / 12 步：组合、美化并交付完整报告**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 误差线与区间图（errorbar）  →  **本章任务：** 子图与组合图（subplots）  →  **下一步：** 美化、注释与导出
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

真实的数据分析几乎不会只靠一张图下结论——我们要同时看销售额趋势、利润变化，还要比较不同渠道的构成。


## 本章目标

学完本章，你将能够：

- **理解**：理解「子图与组合图（subplots）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「子图与组合图（subplots）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「子图与组合图（subplots）」并读出其中的结论。


## 适用场景

**背景引入**：真实的数据分析几乎不会只靠一张图下结论——我们要同时看销售额趋势、利润变化，还要比较不同渠道的构成。子图与组合图（subplots）把多张相关的图表放进同一个画布并共享坐标轴，让我们用一份“整体读图”的视角同时比较它们，避免来回翻页带来的漏读和误判。学会它，你就能把一个分析任务里的多个图表组织成可同步阅读的整体，让对比更直观、结论更有据。 打个比方：子图就像把一本分析报告的几页并排摊开在同一张桌上——它们共用同一把尺（共享坐标轴），你一眼就能横向比较，不用来回翻页；这样趋势、利润、渠道构成放在一起看，结论更有据。

同一分析需要多个互补图表，或需要比较小倍图。


## 数据结构

多个共享维度或相关指标的数据集。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 sharex=True 改为 sharex=False，观察独立坐标轴与共享坐标轴的差异
2. 修改 gridspec_kw 中的 width_ratios 为 [1, 1]，对比均等与非均等列宽布局
3. 调整 figsize 参数（如 (12, 5) 或 (10, 6)），说明画布尺寸对子图可读性的影响


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.grid()`、`fig.suptitle()`、`fig.tight_layout()` | 同一分析需要多个互补图表，或需要比较小倍图。 | 每个子图重复图例和标签 |
| 进阶变体 | `plt.figure()`、`fig.add_gridspec()`、`fig.add_subplot()`、`ax_trend.plot()` | 在基础图表上增加分组、注释、布局或交互 | 子图尺寸太小 |
| 关键参数 | `nrows/ncols` | 网格 | 每个子图重复图例和标签 |
| 关键参数 | `sharex/sharey` | 共享轴 | 子图尺寸太小 |
| 关键参数 | `gridspec_kw` | 比例 | 双Y轴制造虚假同步 |
| 关键参数 | `suptitle` | 画布标题 | 每个子图重复图例和标签 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(8.5, 6), sharex=True)
axes[0].plot(months, sales, marker="o", color="#1a73e8")
axes[0].set(title="销售额", ylabel="万元")
axes[1].plot(months, profit, marker="s", color="#188038")
axes[1].set(title="利润", xlabel="月份", ylabel="万元")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
fig.suptitle("上半年经营指标", fontsize=16)
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：修改子图参数，观察共享轴与独立轴的差异。

37.4 的基础图表用 `sharex=True` 让上下两个子图共享横轴，因此两个子图的 x 轴刻度完全对齐。请把 `sharex=True` 改为 `sharex=False` 再运行，观察第二个子图（利润）的 x 轴如何获得独立的刻度范围；随后把 `nrows=2` 改成 `nrows=1, ncols=2`，看子图从上下排列如何变成左右并列。思考：如果你只想让两张图各自拥有独立的横轴范围，`sharex` 应设为哪个值？


In [ ]:
try:
    # 请在下方填写代码或修改参数
    # 任务：把 sharex=True 改为 sharex=False，观察 x 轴刻度变化；再试 nrows=1, ncols=2
    import matplotlib.pyplot as plt

    # 请修改这一行：将 sharex=True 改为 sharex=False（观察 x 轴独立刻度）

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 5), layout="constrained")
grid = fig.add_gridspec(2, 2, width_ratios=[2, 1])
ax_trend = fig.add_subplot(grid[:, 0])
ax_region = fig.add_subplot(grid[0, 1])
ax_channel = fig.add_subplot(grid[1, 1])
ax_trend.plot(months, sales, marker="o", color="#1a73e8")
ax_trend.set(title="月度销售趋势", ylabel="万元")
ax_region.barh(regions, online + offline, color="#188038")
ax_region.set(title="区域总量")
ax_channel.pie(
    [online.sum(), offline.sum()],
    labels=["线上", "线下"],
    autopct="%.0f%%",
    colors=["#1a73e8", "#f9ab00"],
)
ax_channel.set_title("渠道构成")
plt.show()


## 参数说明

- nrows/ncols：网格
- sharex/sharey：共享轴
- gridspec_kw：比例
- suptitle：画布标题


## 结果解读

先阅读总标题，再按固定顺序逐图比较；共享轴时确认量纲一致。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 每个子图重复图例和标签
- 子图尺寸太小
- 双Y轴制造虚假同步


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把上下两行改成左右两列，对比不同布局的阅读效果
    # 【目标】同一个内容，改成左右并排，练习理解「布局方向影响阅读顺序」。
    import matplotlib.pyplot as plt

    # 起点示例(已可运行)：subplots(1,2) 改为左右并排。
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].plot(months, sales, marker="o", color="#1a73e8")
    axes[0].set(title="销售额", ylabel="万元")
    axes[1].plot(months, profit, marker="s", color="#188038")
    axes[1].set(title="利润", ylabel="万元")
    fig.suptitle("上半年经营指标（并排）", fontsize=16)
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：上下 vs 左右，读图顺序有何不同 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

使用subplots和GridSpec把多个相关图组织为共享阅读结构。


### 你已经掌握

- 判断子图与组合图（subplots）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `nrows/ncols` | 网格 |
| `sharex/sharey` | 共享轴 |
| `gridspec_kw` | 比例 |
| `suptitle` | 画布标题 |


### 需要注意

- 每个子图重复图例和标签
- 子图尺寸太小
- 双Y轴制造虚假同步


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import matplotlib.pyplot as plt

# 完整答案：把 sharex 改为 False，让两个子图各自拥有独立的 x 轴刻度

fig, axes = plt.subplots(2, 1, figsize=(8.5, 6), sharex=False)
axes[0].plot(months, sales, marker="o", color="#1a73e8")
axes[0].set(title="销售额", ylabel="万元")
axes[1].plot(months, profit, marker="s", color="#188038")
axes[1].set(title="利润", xlabel="月份", ylabel="万元")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
fig.suptitle("上半年经营指标", fontsize=16)
fig.tight_layout()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(months, orders, marker="o", color="#1a73e8")
axes[0].set(title="订单趋势")
axes[1].bar(regions, online, color="#188038")
axes[1].set(title="线上销售")
axes[2].hist(samples, bins=14, color="#f9ab00", edgecolor="white")
axes[2].set(title="订单金额分布")
fig.suptitle("经营分析面板")
fig.tight_layout()
plt.show()
